<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 1</b>

## Unit 5: RAG Evaluation

**CV Raman Global University, Bhubaneswar**  
*AI Center of Excellence*

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Poorit-Technologies/cvraman-coe/blob/main/courses-contents/ai-systems-engineering-1/unit-5/01-ai-systems-engineering-1-unit5-rag-evaluation.ipynb)

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Understand RAG evaluation metrics** — what they measure and why they matter
2. **Evaluate retrieval quality** using keyword coverage and MRR
3. **Evaluate answer quality** using the LLM-as-Judge approach
4. **Run a full evaluation pipeline** and interpret the results

**Duration:** ~1 hour

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q openai chromadb sentence-transformers

import os
from getpass import getpass
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb

# Configure API
api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
client = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"

---

## 2. Why Evaluate RAG?

RAG evaluation is critical for building reliable systems:

| Component | What to Measure | Why It Matters |
|-----------|----------------|----------------|
| **Retrieval** | Are relevant docs found? | Bad retrieval = bad answers |
| **Generation** | Is the answer accurate? | LLM can hallucinate |
| **End-to-end** | Does it solve user needs? | Overall system quality |

### Key Metrics

```
Retrieval Metrics:          Answer Metrics:
├── MRR (Mean Reciprocal    ├── Accuracy
│   Rank)                   ├── Completeness
└── Keyword Coverage        └── Relevance
```

For production RAG evaluation at scale, see the [RAGAS framework](https://docs.ragas.io/).

---

## 3. Evaluation Metrics

Before writing any code, let's understand each metric we'll use.

### Keyword Coverage (Retrieval)

The simplest retrieval metric — did the retrieved documents contain the keywords we expected?

| Test Question | Expected Keywords | Keywords Found in Results | Coverage |
|---------------|-------------------|---------------------------|----------|
| "Who founded TechSolutions?" | Priya, Sharma, Rahul, Verma, 2018 | Priya, Sharma, Rahul, Verma, 2018 | **100%** |
| "What is the leave policy?" | 24, paid, 10, sick, leaves | 24, paid, leaves | **60%** |

**Formula:** `keywords_found / total_keywords × 100`

### MRR — Mean Reciprocal Rank (Retrieval)

MRR measures how high the **first relevant document** appears in the search results. The higher it is ranked, the better.

```
Rank of first relevant doc    →    Reciprocal Rank
─────────────────────────────────────────────────
        1st position           →    1/1 = 1.00
        2nd position           →    1/2 = 0.50
        3rd position           →    1/3 = 0.33
        Not found              →    0.00
```

**Worked example:**

Suppose we search for "Who founded TechSolutions?" and get 3 results:
1. "TechSolutions India was founded in 2018 by Priya Sharma..." ← **Relevant!**
2. "Priya Sharma is the CEO and co-founder..."
3. "Major clients include HDFC Bank..."

The first relevant doc is at **rank 1**, so Reciprocal Rank = **1/1 = 1.0**

If the relevant doc appeared at rank 2 instead, the score would be **1/2 = 0.5**

Further reading: [Mean Reciprocal Rank (Wikipedia)](https://en.wikipedia.org/wiki/Mean_reciprocal_rank)

### LLM-as-Judge (Answer Quality)

Instead of manually checking every answer, we use an LLM to rate the generated answer against a reference answer on three dimensions:

| Dimension | What It Measures | Scale |
|-----------|-----------------|-------|
| **Accuracy** | Is the information factually correct? | 1–5 |
| **Completeness** | Does it cover all key points from the reference? | 1–5 |
| **Relevance** | Does it actually answer the question asked? | 1–5 |

This approach is widely used in practice — it's fast, scalable, and correlates well with human judgment.

Further reading: [Judging LLM-as-a-Judge with MT-Bench (Zheng et al.)](https://arxiv.org/abs/2306.05685)

---

## 4. RAG System

A minimal RAG system to evaluate — knowledge base, retriever, and generator.

In [ ]:
# Knowledge base
knowledge_documents = [
    "TechSolutions India was founded in 2018 by Priya Sharma and Rahul Verma in Bhubaneswar, Odisha. The company has grown to over 250 employees.",
    "Priya Sharma is the CEO and co-founder of TechSolutions. She graduated from IIT Delhi and Stanford. She previously worked as Senior Director at Infosys.",
    "Rahul Verma is the CTO and co-founder. He graduated from BITS Pilani and IIM Bangalore. He previously worked as Tech Lead at Google India.",
    "CloudAssist Pro is the flagship product. It costs ₹50,000 per month and includes auto-scaling, monitoring, and 24/7 support.",
    "SmartHR is an AI-powered HR system. It costs ₹25,000 per month and handles recruitment, payroll, and performance tracking.",
    "DataViz Analytics costs ₹15,000 per month. It provides real-time dashboards, custom reports, and predictive analytics.",
    "Leave policy: Employees receive 24 paid leaves and 10 sick leaves per year. Unused leaves can be carried forward up to 10 days.",
    "Health insurance coverage of ₹5 lakh for employees and families. Performance bonus up to 20% of annual salary.",
    "Work hours are 9 AM to 6 PM, Monday to Friday. Hybrid model with 3 days office, 2 days remote.",
    "Major clients include HDFC Bank, Tata Motors, Reliance Industries, and ICICI Bank."
]

# Initialize embedding model and ChromaDB
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma = chromadb.Client()

try:
    chroma.delete_collection("eval_test")
except:
    pass

collection = chroma.create_collection("eval_test")

# Index documents
embeddings = embedder.encode(knowledge_documents)
collection.add(
    ids=[f"doc_{i}" for i in range(len(knowledge_documents))],
    embeddings=embeddings.tolist(),
    documents=knowledge_documents
)

def retrieve(query, n_results=3):
    """Retrieve relevant documents."""
    query_embedding = embedder.encode([query])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    return results['documents'][0]

def generate_answer(query, context):
    """Generate answer from context."""
    system = f"""Answer based on context. Be accurate and concise.
    
Context:
{chr(10).join(context)}"""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": query}
        ],
        temperature=0
    )
    return response.choices[0].message.content

---

## 5. Test Cases

We need test cases to evaluate our RAG system. Each test case has a question, the expected answer, and keywords we expect the retrieval to find.

In [ ]:
test_cases = [
    {
        "question": "Who founded TechSolutions India?",
        "expected_answer": "Priya Sharma and Rahul Verma founded TechSolutions India in 2018.",
        "keywords": ["Priya", "Sharma", "Rahul", "Verma", "2018"]
    },
    {
        "question": "How much does CloudAssist Pro cost?",
        "expected_answer": "CloudAssist Pro costs ₹50,000 per month.",
        "keywords": ["50,000", "month", "CloudAssist"]
    },
    {
        "question": "What is the leave policy at TechSolutions?",
        "expected_answer": "Employees get 24 paid leaves and 10 sick leaves per year.",
        "keywords": ["24", "paid", "10", "sick", "leaves"]
    }
]

---

## 6. Retrieval Evaluation

We'll measure two things about our retrieval:
- **Keyword Coverage** — what percentage of expected keywords appear in retrieved docs?
- **MRR** — how high is the first relevant document ranked?

Keyword coverage and MRR are complementary: coverage measures breadth (were the right terms found?), while MRR measures ranking quality (was the best document ranked first?).

In [ ]:
def evaluate_retrieval(question, keywords, n_results=3):
    """Evaluate retrieval quality for a question."""
    docs = retrieve(question, n_results=n_results)
    combined_text = " ".join(docs).lower()

    # Keyword coverage
    found = sum(1 for kw in keywords if kw.lower() in combined_text)
    coverage = found / len(keywords) * 100

    # MRR — find first doc containing any keyword
    mrr = 0.0
    for rank, doc in enumerate(docs, 1):
        if any(kw.lower() in doc.lower() for kw in keywords):
            mrr = 1.0 / rank
            break

    return {"mrr": mrr, "coverage": coverage, "docs": docs}

---

## 7. LLM-as-Judge

Now we evaluate the **generated answers**. We send the question, reference answer, and generated answer to an LLM, which rates the answer on accuracy, completeness, and relevance (1–5 scale).

This approach works because LLMs can assess factual consistency and completeness at scale, avoiding the bottleneck of manual review. Research shows LLM-as-Judge ratings correlate well with human judgment, though it's important to be aware of potential biases such as preferring longer or more verbose answers.

Further reading: [Judging LLM-as-a-Judge with MT-Bench (Zheng et al.)](https://arxiv.org/abs/2306.05685)

In [ ]:
EVAL_PROMPT = """
Evaluate this answer against the reference.

Question: {question}
Reference Answer: {reference}
Generated Answer: {answer}

Rate on a scale of 1-5:
- Accuracy: Is the information correct?
- Completeness: Does it cover all key points?
- Relevance: Does it answer the question?

Respond in this exact format:
ACCURACY: [1-5]
COMPLETENESS: [1-5]
RELEVANCE: [1-5]
FEEDBACK: [One sentence explanation]
"""

def evaluate_answer(question, expected_answer, generated_answer):
    """Evaluate answer quality using LLM-as-Judge."""
    prompt = EVAL_PROMPT.format(
        question=question,
        reference=expected_answer,
        answer=generated_answer
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    # Parse scores from response
    text = response.choices[0].message.content
    scores = {}
    feedback = ""
    for line in text.strip().split("\n"):
        for key in ["ACCURACY", "COMPLETENESS", "RELEVANCE"]:
            if line.startswith(key + ":"):
                scores[key.lower()] = float(line.split(":")[1].strip())
        if line.startswith("FEEDBACK:"):
            feedback = line.split(":", 1)[1].strip()

    return {
        "accuracy": scores.get("accuracy", 0),
        "completeness": scores.get("completeness", 0),
        "relevance": scores.get("relevance", 0),
        "feedback": feedback
    }

---

## 8. Run Evaluation

Run retrieval and answer evaluation together.

In [ ]:
results = []

for test in test_cases:
    ret = evaluate_retrieval(test["question"], test["keywords"])
    answer = generate_answer(test["question"], ret["docs"])
    scores = evaluate_answer(test["question"], test["expected_answer"], answer)
    results.append({**ret, **scores})

    print(f"Q: {test['question']}")
    print(f"  Retrieval — MRR: {ret['mrr']:.2f}, Coverage: {ret['coverage']:.0f}%")
    print(f"  Answer    — Acc: {scores['accuracy']}/5, Comp: {scores['completeness']}/5, Rel: {scores['relevance']}/5\n")

n = len(results)
print(f"Avg Retrieval — MRR: {sum(r['mrr'] for r in results)/n:.2f}, Coverage: {sum(r['coverage'] for r in results)/n:.0f}%")
print(f"Avg Answer    — Acc: {sum(r['accuracy'] for r in results)/n:.1f}, Comp: {sum(r['completeness'] for r in results)/n:.1f}, Rel: {sum(r['relevance'] for r in results)/n:.1f}")

---

## Key Takeaways

1. **Evaluation is critical** - can't improve what you can't measure

2. **Measure both retrieval and generation** - they fail differently

3. **Use multiple metrics** - no single metric tells the whole story

4. **LLM-as-judge works** - for answer quality evaluation

### Evaluation Metrics Summary

| Metric | Measures | Good Value |
|--------|----------|------------|
| MRR | Rank of first relevant doc | > 0.5 |
| Keyword Coverage | Keywords found in results | > 80% |
| Accuracy | Correctness of answer | 4-5 |
| Completeness | Coverage of key points | 4-5 |
| Relevance | Answer fits question | 4-5 |

### What's Next?

In the next notebook, we'll learn advanced RAG techniques like reranking and query rewriting.

---

## Resources

- [RAGAS Framework](https://docs.ragas.io/) — production-grade RAG evaluation
- [Judging LLM-as-a-Judge with MT-Bench (Zheng et al.)](https://arxiv.org/abs/2306.05685) — foundational paper on LLM-based evaluation
- [LangChain Evaluation](https://python.langchain.com/docs/guides/productionization/evaluation/) — evaluation tools in LangChain
- [Mean Reciprocal Rank (Wikipedia)](https://en.wikipedia.org/wiki/Mean_reciprocal_rank) — MRR metric explained

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 1
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---